In [5]:
import openai

In [7]:
def summarize_and_classify(client, title: str, text: str, categories: list[str]) -> dict:
    """
    요구사항:
    - client.messages.create()로 Claude(또는 OpenAI) API를 호출한다.
    - 프롬프트는 "요약: <2문장 요약>" / "카테고리: <categories 중 하나>" 형식으로만
      답하도록 명시적으로 지시한다.
    - 응답 문자열에서 "요약:"/"카테고리:" 줄을 찾아 파싱한다.
    - 반환값: {"title": ..., "summary": ..., "category": ...}
    - 파싱 실패 시(접두어를 못 찾으면) category는 "Other"로 fallback한다.
    """
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant that summarizes text and classifies it into one of the provided categories. "
                    "Please respond in the following format:\n"
                    "그 외의 설명, 인사말, 마크다운은 무시하고, 반드시 아래 형식으로만 답변\n"
                    "요약: <2문장 요약>\n"
                    "카테고리: <categories 중 하나>"
                ),
            },
            {
                "role": "user",
                "content": f"Title: {title}\nText: {text}\nCategories: {', '.join(categories)}",
            },
        ])
    response_text = response.choices[0].message.content
    response_lines = response_text.splitlines()
    summary = None
    category = "Other"  # Default fallback category
    for line in response_lines:
        line = line.strip()
        if line.startswith("요약:"):
            summary = line[len("요약:"):].strip()
        elif line.startswith("카테고리:"):
            category = line[len("카테고리:"):].strip()
    return {"title": title, "summary": summary, "category": category}
client = openai.OpenAI()  # Initialize your client here
title = "Sample Title"
text = "This is a sample text that needs to be summarized and classified into a category."
categories = ["Technology", "Science", "Health", "Entertainment"]
summarize_and_classify(client, title, text, categories)

{'title': 'Sample Title',
 'summary': '이 텍스트는 요약하고 분류해야 하는 샘플 텍스트이다. 구체적인 내용은 제공되지 않았다.',
 'category': 'Entertainment'}